In [1]:
import os

from google import genai
from google.genai import types
from google.genai import errors

# Set GEMINI_API_KEY and STOCK_TICKER in the environment before running this notebook.
symbol = os.environ.get("STOCK_TICKER", "").strip().upper()
if not symbol:
    raise RuntimeError("Set STOCK_TICKER before running enterprise.ipynb")

api_key = os.getenv("GEMINI_API_KEY", "").strip()
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY before running enterprise.ipynb")

client = genai.Client(api_key=api_key)

# Ordered list of models for fallback to handle rate limits
fallback_models = [
    "gemini-3.8-flash",
    "gemini-3.0-flash",       # Standardized "Gemini 3 Flash"
    "gemini-2.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",
    "gemini-2.5-flash-lite"
]

description_response = None
model_errors = []

for model_name in fallback_models:
    try:
        description_response = client.models.generate_content(
            model=model_name,
            contents=(
                f"Describe the company identified by stock ticker {symbol}. "
                "If the ticker is unknown or ambiguous, say so instead of guessing."
            ),
            config=types.GenerateContentConfig(
                system_instruction="Answer professionally in no more than 200 words. Describe the company's business concisely, without bullet points. You may include historical facts.",
                temperature=0.2,
            ),
        )
        break
        
    except errors.APIError as e:
        model_errors.append(f"{model_name}: {e}")
        continue
    except Exception as e:
        model_errors.append(f"{model_name}: {e}")
        continue

# Check if we exhausted all models without getting a response
if description_response is None:
    error_details = "\n".join(model_errors)
    raise RuntimeError(
        f"Failed to retrieve a description for {symbol} after exhausting all fallback models.\n"
        f"Model errors:\n{error_details}"
    )

description = (description_response.text or "").strip()
if not description:
    raise RuntimeError(f"Gemini returned an empty description for {symbol}")

print(description)

Broadcom Inc., trading under the ticker AVGO, is a global technology leader that designs, develops, and supplies a broad range of semiconductor and infrastructure software solutions. The company’s core business encompasses enterprise software, data center networking, broadband, wireless, and industrial products that serve the networking, telecommunications, storage, and industrial markets. 

The company's origins trace back to Hewlett-Packard’s semiconductor division, which was spun off in 1999 as Agilent Technologies. Agilent's semiconductor business was subsequently acquired in 2005 by private equity firms and renamed Avago Technologies. In 2016, Avago acquired the original Broadcom Corporation and adopted the iconic Broadcom name, subsequently executing major strategic expansions. These included the 2018 acquisition of CA Technologies, the 2019 purchase of Symantec’s enterprise security business, and the landmark acquisition of VMware in 2023. Headquartered in Palo Alto, California,